round 2

In [1]:
import numpy as np
budget = 50000

research_spend = 0
scale_spend = 0
speed_spend = 0

total_spend = research_spend + scale_spend + speed_spend

research_pct = research_spend / 50000
scale_pct = scale_spend / 50000
speed_pct = speed_spend / 50000

research_factor = 200000 * np.log(1 + research_pct) / np.log(1 + 100)
scale_factor = 0.07 * scale_pct
speed_factor = 0.5

profit = research_factor * scale_factor * speed_factor - total_spend

In [2]:
import numpy as np
from scipy.optimize import brentq

def optimal_r(v):
    B = 50000 - v
    f = lambda r: (B - r) / (500 + r) - np.log(1 + r/500)
    return brentq(f, 0, B)



speed_pcts = np.arange(30, 50, 1)
for speed_pct in speed_pcts:
    speed_spend = speed_pct * 500
    r_pct = optimal_r(speed_spend)/500
    r_fact = 200000 * np.log(1+r_pct) / np.log(101)
    s_pct = (50000 - speed_spend - optimal_r(speed_spend) ) / 500
    s_fact = 0.07 * s_pct

    # assume median speed is 33% allocation
    median_speed = 33
    est_speed_factor = speed_pct / (median_speed * 2)
    est_speed_factor = min(max(0.1, est_speed_factor), 0.9)

    print(f'speed: {speed_pct} at {round(est_speed_factor, 2)} optimal r: {round(r_pct, 2)}, optimal s: {round(s_pct, 2)}')
    print(f'profit here: { round(r_fact * s_fact * est_speed_factor - 50000, 2)}')


speed: 30 at 0.45 optimal r: 17.2, optimal s: 52.8
profit here: 161235.58
speed: 31 at 0.47 optimal r: 16.99, optimal s: 52.01
profit here: 164150.85
speed: 32 at 0.48 optimal r: 16.79, optimal s: 51.21
profit here: 166816.65
speed: 33 at 0.5 optimal r: 16.58, optimal s: 50.42
profit here: 169234.79
speed: 34 at 0.52 optimal r: 16.38, optimal s: 49.62
profit here: 171407.1
speed: 35 at 0.53 optimal r: 16.17, optimal s: 48.83
profit here: 173335.46
speed: 36 at 0.55 optimal r: 15.97, optimal s: 48.03
profit here: 175021.8
speed: 37 at 0.56 optimal r: 15.76, optimal s: 47.24
profit here: 176468.07
speed: 38 at 0.58 optimal r: 15.55, optimal s: 46.45
profit here: 177676.28
speed: 39 at 0.59 optimal r: 15.34, optimal s: 45.66
profit here: 178648.47
speed: 40 at 0.61 optimal r: 15.13, optimal s: 44.87
profit here: 179386.72
speed: 41 at 0.62 optimal r: 14.92, optimal s: 44.08
profit here: 179893.18
speed: 42 at 0.64 optimal r: 14.71, optimal s: 43.29
profit here: 180170.03
speed: 43 at 0.65

round 3 manual

In [3]:
def cdf_bid(x):
    val = ((x-670) // 5)
    return (1/51) * (val+1)

def profit_func(low_bid, high_bid, mean):
    scaling = np.where(high_bid < mean, ((920 - mean) / (920 - high_bid)) ** 3 ,1)

    return cdf_bid(low_bid) * (920 - low_bid) + (cdf_bid(high_bid) - cdf_bid(low_bid)) * (920-high_bid) * scaling

# Grid search to find best bids
def best_bids(mu):

    low_bids = np.arange(670, 920)
    high_bids = np.arange(670, 920)
    X, Y = np.meshgrid(low_bids, high_bids)
    mean = mu

    Z = profit_func(X, Y, mean)



    # Find the maximum profit in the grid
    max_index = np.unravel_index(np.argmax(Z, axis=None), Z.shape)
    opt_L = X[max_index]
    opt_H = Y[max_index]
    max_profit = Z[max_index]

    return {
        'opt_L': opt_L,
        'opt_H': opt_H,
        'max_profit': max_profit,
    }

In [4]:
best_bids(838)

{'opt_L': np.int64(750),
 'opt_H': np.int64(840),
 'max_profit': np.float64(84.90196078431373)}

In [5]:
def calc_optimal_investment(est_changes):
    for product in est_changes:
        pct_change = est_changes[product]
        profits = []

        if pct_change >= 0:
            for i in range(100):
                investment = i*10000
                net_profit = investment * pct_change - ((investment / 1000000) ** 2) * 1000000
                profits.append(net_profit)
            best_invest = np.argmax(profits)
            print(f'optimal investment for {product}: {best_invest} (LONG)')
        else:
            for i in range(100):
                investment = i*10000
                net_profit = investment * (-pct_change) - ((investment / 1000000) ** 2) * 1000000
                profits.append(net_profit)
            best_invest = np.argmax(profits)
            print(f'optimal investment for {product}: {best_invest} (SHORT)')

pct_changes = {
    "obsidian cutlery": 0.35,
    "pyroflex cells": 0,
    "thermalite core": 0.25,
    "lava cake": -0.4,
    "magma ink": 0.15,
    "scoria paste": 0.05,
    "ashes of the phoenix": -0.3,
    "volcanic incense": 0.05,
    "sulfur reactor": 0.30,
}

calc_optimal_investment(pct_changes)

optimal investment for obsidian cutlery: 18 (LONG)
optimal investment for pyroflex cells: 0 (LONG)
optimal investment for thermalite core: 12 (LONG)
optimal investment for lava cake: 20 (SHORT)
optimal investment for magma ink: 8 (LONG)
optimal investment for scoria paste: 2 (LONG)
optimal investment for ashes of the phoenix: 15 (SHORT)
optimal investment for volcanic incense: 2 (LONG)
optimal investment for sulfur reactor: 15 (LONG)
